In [1]:
import sys, os
sys.path.append('..')

In [2]:
import pandas as pd
import re

In [3]:
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *

In [ ]:
from pymatgen.core.composition import Composition

def update_result(df):
    def count_atoms(formula):
        try:
            comp = Composition(formula)
            return comp.num_atoms
        except Exception:
            return None

    df['atom_count'] = df['pretty_formula'].apply(count_atoms)
    return df

def read_results(file, split_file):
    rows = []
    pattern = r'^(?P<status>E|x)?\s*(?P<idx>\d+):\s*(?P<material>[a-zA-Z]+-\d+)\s*---\s*(?P<time>[\d\.eE+-]+)'

    for line in open(file).read().strip().split('\n'):
        m = re.match(pattern, line)
        if m:
            rows.append({
                'status': m.group('status') or '',
                'idx': int(m.group('idx')),
                'material_id': m.group('material'),
                'time': float(m.group('time')),
            })

    df = pd.DataFrame(rows)

    split = read_split_df(split_file)

    result = pd.merge(df, split, on='material_id', how='inner')

    return update_result(result)

In [6]:
tests = read_results("results.txt", "test")
tests

,status,idx,material_id,time,Unnamed: 0,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number,atom_count
0,E,0,mp-10009,0.179174,6000,-0.575092,0.8980,GaTe,0.000000,"['Ga', 'Te']",# generated using pymatgen\ndata_GaTe\n_symmet...,194,2.0
1,E,1,mp-1218989,0.137652,37702,-0.942488,0.0000,SmThCN,0.044109,"['C', 'N', 'Sm', 'Th']",# generated using pymatgen\ndata_SmThCN\n_symm...,160,4.0
2,x,2,mp-1225695,1.188268,42245,0.064863,0.0000,CuNi,0.064863,"['Cu', 'Ni']",# generated using pymatgen\ndata_CuNi\n_symmet...,65,2.0
3,,3,mp-1220884,1.707755,780,-1.456116,0.0000,NaTiVS4,0.000000,"['Na', 'S', 'Ti', 'V']",# generated using pymatgen\ndata_NaTiVS4\n_sym...,8,7.0
4,,4,mp-1224266,2.922140,35749,0.024139,0.0000,Ho3TmMn8,0.036496,"['Ho', 'Mn', 'Tm']",# generated using pymatgen\ndata_Ho3TmMn8\n_sy...,8,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9041,,9041,mp-21084,3.431612,7763,-1.777752,1.0134,In6Ga2PtO8,0.000000,"['Ga', 'In', 'O', 'Pt']",# generated using pymatgen\ndata_In6Ga2PtO8\n_...,225,17.0
9042,,9042,mp-571486,4.601307,15377,-0.359373,0.0000,CuSe,0.000000,"['Cu', 'Se']",# generated using pymatgen\ndata_CuSe\n_symmet...,63,2.0
9043,,9043,mp-14410,6.725072,17730,-1.205080,0.4594,Tl6TeO12,0.000000,"['O', 'Te', 'Tl']",# generated using pymatgen\ndata_Tl6TeO12\n_sy...,148,19.0
9044,,9044,mp-1079192,1.939557,28030,-2.814520,0.0000,Sr2GdRuO6,0.014361,"['Gd', 'O', 'Ru', 'Sr']",# generated using pymatgen\ndata_Sr2GdRuO6\n_s...,87,10.0


In [4]:
results = pd.read_csv('mp20-results.csv', delimiter=',')
results = results.fillna('0')
results = results[results.status == '0']
results

,Unnamed: 0,status,material_id,time
2,2,0,mp-1209679,0.789871
3,3,0,mp-1020059,0.620733
4,4,0,mp-1189429,1.205847
5,5,0,mp-756309,2.557689
6,6,0,mp-1220565,1.129870
...,...,...,...,...
45146,3180,0,mp-1106031,66.960862
45150,15520,0,mp-677378,67.381793
45159,1538,0,mp-1228959,68.343451
45210,12602,0,mp-644506,77.162810


In [5]:
mp_20 = pd.concat([read_split_df('train'), read_split_df('test'), read_split_df('val')])
mp_20

,Unnamed: 0,material_id,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number
0,37228,mp-1221227,-1.637460,0.2133,Na3MnCoNiO6,0.043001,"['Co', 'Mn', 'Na', 'Ni', 'O']",# generated using pymatgen\ndata_Na3MnCoNiO6\n...,8
1,19480,mp-974729,-0.314759,0.0000,Nd(Al2Cu)4,0.000000,"['Al', 'Cu', 'Nd']",# generated using pymatgen\ndata_Nd(Al2Cu)4\n_...,139
2,29624,mp-1185360,-0.193761,0.0000,LiMnIr2,0.018075,"['Ir', 'Li', 'Mn']",# generated using pymatgen\ndata_LiMnIr2\n_sym...,225
3,38633,mp-1188861,-0.584694,3.8556,LiCSN,0.048847,"['C', 'Li', 'N', 'S']",# generated using pymatgen\ndata_LiCSN\n_symme...,62
4,10889,mp-677272,-2.474759,0.4707,La2EuS4,0.000000,"['Eu', 'La', 'S']",# generated using pymatgen\ndata_La2EuS4\n_sym...,122
...,...,...,...,...,...,...,...,...,...
9042,21009,mp-1023925,-1.152798,1.6970,WS2,0.001173,"['S', 'W']",# generated using pymatgen\ndata_WS2\n_symmetr...,164
9043,31626,mp-1187764,-0.788772,0.0000,Y2ZnPt,0.022813,"['Pt', 'Y', 'Zn']",# generated using pymatgen\ndata_Y2ZnPt\n_symm...,225
9044,20673,mp-1219588,-2.910913,1.9239,RbMgCoF6,0.000710,"['Co', 'F', 'Mg', 'Rb']",# generated using pymatgen\ndata_RbMgCoF6\n_sy...,74
9045,5349,mp-3589,-2.764644,7.2758,BPO4,0.000000,"['B', 'P', 'O']",# generated using pymatgen\ndata_BPO4\n_symmet...,82


In [36]:
import ast

In [38]:
# mp_20[(len(mp_20.elements) > 1) & (mp_20.band_gap == 0)]
# mp_20[(mp_20.formation_energy_per_atom == 0) & (len(mp_20.elements) > 2)]
mp_20[(mp_20.formation_energy_per_atom == 0) & mp_20.elements.apply(ast.literal_eval).apply(lambda x: len(x) > 1)]

,Unnamed: 0,material_id,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number


In [6]:
joined = results.merge(mp_20, on='material_id', how='left')
# joined[(joined.formation_energy_per_atom != 0) & (joined.band_gap != 0)]
joined = joined[joined.formation_energy_per_atom != 0]
joined

,Unnamed: 0_x,status,material_id,time,Unnamed: 0_y,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number
0,2,0,mp-1209679,0.789871,8138,-0.711824,0.0000,PrBiRh,0.000000,"['Bi', 'Pr', 'Rh']",# generated using pymatgen\ndata_PrBiRh\n_symm...,62
1,3,0,mp-1020059,0.620733,5513,-0.509370,2.7580,LiGe2N3,0.000000,"['Li', 'Ge', 'N']",# generated using pymatgen\ndata_LiGe2N3\n_sym...,36
2,4,0,mp-1189429,1.205847,11129,-0.875173,0.0000,LaGaPd2,0.000000,"['Ga', 'La', 'Pd']",# generated using pymatgen\ndata_LaGaPd2\n_sym...,62
3,5,0,mp-756309,2.557689,34432,-3.919080,0.0000,Ce4DyO9,0.031046,"['Ce', 'Dy', 'O']",# generated using pymatgen\ndata_Ce4DyO9\n_sym...,44
4,6,0,mp-1220565,1.129870,41217,0.006919,0.0000,Nd(Co5Mo)2,0.060941,"['Co', 'Mo', 'Nd']",# generated using pymatgen\ndata_Nd(Co5Mo)2\n_...,71
...,...,...,...,...,...,...,...,...,...,...,...,...
38451,3180,0,mp-1106031,66.960862,39564,-2.292986,0.0000,NaMn3V4O12,0.051950,"['Mn', 'Na', 'O', 'V']",# generated using pymatgen\ndata_NaMn3V4O12\n_...,204
38452,15520,0,mp-677378,67.381793,36371,-3.009293,2.5927,TiNbTl(O2F)2,0.038870,"['F', 'Nb', 'O', 'Ti', 'Tl']",# generated using pymatgen\ndata_TiNbTl(O2F)2\...,9
38453,1538,0,mp-1228959,68.343451,35777,-3.405102,1.5705,CsPr2Ti2NbO10,0.036198,"['Cs', 'Nb', 'O', 'Pr', 'Ti']",# generated using pymatgen\ndata_CsPr2Ti2NbO10...,8
38454,12602,0,mp-644506,77.162810,15398,-2.183420,5.0934,NaAlH2CO5,0.000000,"['Al', 'C', 'H', 'Na', 'O']",# generated using pymatgen\ndata_NaAlH2CO5\n_s...,74


In [7]:
from slices.core import SLICES
from pymatgen.core.structure import Structure
from multiprocessing import Process, Queue
from pymatgen.io.cif import CifParser
from io import StringIO
from tqdm import tqdm
import csv

def collect_into_trainset(dataset, save_file):
    backend = SLICES(relax_model="a")
    # data = []
    with open(save_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['material_id', 'band_gap', 'formation_energy', 'slices_string'])
        for row in tqdm(dataset.iterrows(), total=dataset.shape[0], smoothing=0.05):
            r = row[1]
            slices = backend.structure2SLICES(CifParser(StringIO(r.cif)).parse_structures()[0], strategy=3)
            writer.writerow([r.material_id, r.band_gap, r.formation_energy_per_atom, slices])
            # data.append(f'<BOG> <GB_{row.band_gap}> <FE_{row.formation_energy_per_atom}> | {slices} <EOS>')
    
        # for line in data:
        #     f.write(line + '\n')

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [8]:
collect_into_trainset(joined, "mp-20-slices.txt")

  0%|          | 0/38396 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
  0%|          | 5/38396 [00:00<1:31:15,  7.01it/s]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
  0%|          | 14/38396 [00:01<1:14:00,  8.64it/s]/usr/local/lib/python3.10/dist-packages/pymatgen/io/cif.py:1219: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
input_txt_path = "mp-20-slices.txt"
df = pd.read_csv(input_txt_path, sep=None, engine='python')

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

train_df["band_gap"] = df["band_gap"].round(1)
train_df["formation_energy"] = df["formation_energy"].round(1)
val_df["band_gap"] = df["band_gap"].round(1)
val_df["formation_energy"] = df["formation_energy"].round(1)

train_df.to_csv("MP20_train_with_slices.csv", index=False)
val_df.to_csv("MP20_val_with_slices.csv", index=False)


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30716 entries, 3031 to 15795
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   material_id       30716 non-null  object 
 1   band_gap          30716 non-null  float64
 2   formation_energy  30716 non-null  float64
 3   slices_string     30716 non-null  object 
dtypes: float64(2), object(2)
memory usage: 1.2+ MB


In [8]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7680 entries, 32555 to 22710
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   material_id       7680 non-null   object 
 1   band_gap          7680 non-null   float64
 2   formation_energy  7680 non-null   float64
 3   slices_string     7680 non-null   object 
dtypes: float64(2), object(2)
memory usage: 300.0+ KB
